# Practice 21 — pandas + NumPy
Work through each task in order. Try to solve it yourself before looking anything up!

In [1]:
import pandas as pd
import numpy as np

---
## Level 1 — Basics

### Task 1: DateTime + MultiIndex
A weather station readings DataFrame has been created for you.

1. Convert `date` to datetime. Extract `year` and `month` as new columns.
2. Add `days_ago` = days between `date` and `2026-03-19`. Find mean and median with NumPy.
3. Set a `(city, station)` MultiIndex. Sort it.
4. Retrieve all rows for `'Miami'`.
5. Use `pd.IndexSlice` to get all `'S'` (south station) rows across every city.

In [8]:
# Starter data — don't change this
readings = pd.DataFrame({
    'city':     ['Austin', 'Austin', 'Denver', 'Denver', 'Miami', 'Miami', 'Phoenix', 'Phoenix'],
    'station':  ['N', 'S', 'N', 'S', 'N', 'S', 'N', 'S'],
    'date':     ['2024-03-15', '2024-09-20', '2024-02-10', '2024-11-05',
                 '2024-06-18', '2025-01-30', '2024-04-22', '2025-02-14'],
    'temp_f':   [72, 95, 38, 55, 88, 76, 85, 62],
    'humidity': [65, 70, 30, 45, 88, 72, 25, 40],
})

# Your code here

readings['date'] = pd.to_datetime(readings['date'])
readings['year'] = readings['date'].dt.year
readings['month'] = readings['date'].dt.month
readings['days_ago'] = (pd.to_datetime('2026-03-19')-readings['date']).dt.days
np.mean(readings['days_ago'])
np.median(readings['days_ago'])
readings = readings.set_index(['city','station']).sort_index()
readings.loc['Miami']
idx = pd.IndexSlice
readings.loc[idx[:,'S'],:]

,,date,temp_f,humidity,year,month,days_ago
city,station,,,,,,
Austin,S,2024-09-20,95,70,2024,9,545
Denver,S,2024-11-05,55,45,2024,11,499
Miami,S,2025-01-30,76,72,2025,1,413
Phoenix,S,2025-02-14,62,40,2025,2,398


---
## Level 2 — Transformations

### Task 2: .pipe()
Use `.pipe(fn)` to apply a function to a DataFrame as part of a method chain — the function receives the DataFrame as its first argument.

A student performance DataFrame has been created for you, along with three helper functions.

1. Use `.pipe()` to apply all three in order: `add_grade`, `add_efficiency`, `flag_at_risk`.
2. How many students are flagged at risk? Use the `at_risk` column.
3. What is the mean `efficiency`? Use NumPy.
4. Which subject has the highest mean `score`? Use groupby and `.idxmax()`.

In [14]:
# Starter data — don't change this
np.random.seed(7)
students = pd.DataFrame({
    'student_id':    [f'S{i:03d}' for i in range(1, 9)],
    'subject':       np.random.choice(['Math', 'Science', 'English'], size=8),
    'score':         np.random.randint(50, 100, size=8),
    'hours_studied': np.random.randint(1, 20, size=8),
    'attendance':    np.random.randint(60, 100, size=8),
})

def add_grade(df):
    df['grade'] = pd.cut(df['score'], bins=[0, 59, 69, 79, 89, 100],
                         labels=['F', 'D', 'C', 'B', 'A'])
    return df

def add_efficiency(df):
    df['efficiency'] = df['score'] / df['hours_studied']
    return df

def flag_at_risk(df):
    df['at_risk'] = (df['score'] < 65) | (df['attendance'] < 75)
    return df

# Your code here
res = (
    students
    .pipe(add_grade)
    .pipe(add_efficiency)
    .pipe(flag_at_risk)
)
res
res['at_risk'].sum()
np.mean(res['efficiency'])
res.groupby('subject')['score'].mean().idxmax()

'Science'

### Task 3: stack() / unstack() / .xs()
A hotel occupancy DataFrame has been created for you (hotels as index, quarters as columns).

1. Stack to long format. Store as `occ_long` (use `.to_frame('occupancy')`)
2. Add `log_occ` using NumPy
3. Add `high_season`: `True` if `occupancy > 75` (use `np.where`)
4. Use `.xs()` to extract all `'Q2'` rows across every hotel
5. Unstack back to wide format
6. Find the hotel with the highest mean occupancy using NumPy

In [47]:
# Starter data — don't change this
np.random.seed(42)
hotels = pd.DataFrame({
    'hotel': ['Grand', 'Plaza', 'Hilltop', 'Harbor'],
    'Q1':    np.random.randint(55, 95, size=4),
    'Q2':    np.random.randint(55, 95, size=4),
    'Q3':    np.random.randint(55, 95, size=4),
    'Q4':    np.random.randint(55, 95, size=4),
}).set_index('hotel')

# Your code here
occ_long = hotels.stack().to_frame('occupancy')
occ_long['log_occ'] = np.log(occ_long['occupancy'])
occ_long['high_season'] = np.where(occ_long['occupancy']>75, True, False)
occ_long.xs('Q2', level=1)
w = occ_long.unstack()
occ_long.groupby('hotel')['occupancy'].mean().idxmax()
m = occ_long.groupby('hotel')['occupancy'].mean()
m.index[np.argmax(m)]
#occ_long


'Grand'

### Task 4: .apply()
A subscription service DataFrame has been created for you.

1. Add `lifetime_value`: row-wise `plan_price * months_sub` using `apply` with `axis=1`
2. Add `plan_tier`: apply a lambda to `plan_price` — `'Basic'` if < 12, `'Standard'` if < 18, else `'Premium'`
3. Add `churn_risk` (row-wise): `'High'` if `logins_pw < 2` **and** `months_sub < 6`, `'Medium'` if either is true, else `'Low'`

In [27]:
# Starter data — don't change this
np.random.seed(33)
subs = pd.DataFrame({
    'member_id':  [f'M{i:03d}' for i in range(1, 9)],
    'plan_price': [9.99, 14.99, 9.99, 19.99, 14.99, 9.99, 19.99, 14.99],
    'months_sub': np.random.randint(1, 36, size=8),
    'logins_pw':  np.random.randint(0, 15, size=8),
})

# Your code here
subs['lifetime_value'] = subs.apply(lambda row: row['plan_price']* row['months_sub'], axis = 1)
subs['plan_tier'] = subs['plan_price'].apply(lambda x: 'Basic' if x<12
                                             else 'Standard' if x <18
                                             else 'Premium')
subs['churn_risk']= subs.apply(lambda row: 'High' if row['logins_pw']<2 and row['months_sub']<6
                               else 'Medium' if row['logins_pw']<2 or row['months_sub']<6
                               else 'Low', axis=1)

### Task 5: .str + .rank()
A restaurant menu DataFrame has been created for you.

1. Add `item_upper`: item names in uppercase
2. Filter to items where `item_name` contains `'the'` (case-insensitive)
3. Extract the numeric part of `item_code` (e.g. `'I05'` → `'05'`) using `.str` slicing. Store as `num`.
4. Add `price_rank`: rank by `price`, most expensive first
5. Add `category_rating_rank`: rank within each `category` by `rating`, highest first
6. Filter to the top-ranked item per category (`category_rating_rank == 1`)

In [46]:
# Starter data — don't change this
np.random.seed(15)
menu = pd.DataFrame({
    'item_code': [f'I{i:02d}' for i in range(1, 9)],
    'item_name': ['Grilled Salmon', 'Caesar Salad', 'The Classic Burger',
                  'Margherita Pizza', 'The Steak', 'Pasta Primavera',
                  'Fish Tacos', 'The Club Sandwich'],
    'category':  ['Mains', 'Starters', 'Mains', 'Mains', 'Mains', 'Mains', 'Mains', 'Starters'],
    'price':     np.round(np.random.uniform(8, 35, size=8), 2),
    'rating':    np.round(np.random.uniform(3.0, 5.0, size=8), 2),
})

# Your code here
menu['item_upper'] = menu['item_name'].str.upper()
menu[menu['item_name'].str.contains('the', case=False)]
menu['num'] = menu['item_code'].str[1:]
menu['price_rank'] = menu['price'].rank(ascending=False)
menu['category_rating_rank'] = menu.groupby('category')['rating'].rank(ascending=False)
menu[menu['category_rating_rank']==1]


,item_code,item_name,category,price,rating,item_upper,num,price_rank,category_rating_rank
1,I02,Caesar Salad,Starters,12.83,3.50,CAESAR SALAD,02,7.0,1.0
2,I03,The Classic Burger,Mains,9.47,4.84,THE CLASSIC BURGER,03,8.0,1.0


---
## Level 3 — Aggregation

### Task 6: pd.merge() + pd.concat() + pivot_table
Two DataFrames have been created for you: `doctors` and `appointments`.

1. Inner join `appointments` with `doctors` on `doctor_id`
2. Add `dept_avg_wait`: mean `wait_mins` per `dept` using `transform`
3. Add `above_avg_wait`: `True` if `wait_mins > dept_avg_wait`
4. Two separate feedback DataFrames (`feedback_h1`, `feedback_h2`) have also been created. Concatenate vertically, reset the index. Store as `feedback`.
5. Pivot `feedback`: mean `rating` by `dept` (rows) and `half` (columns)

In [33]:
# Starter data — don't change this
np.random.seed(91)
doctors = pd.DataFrame({
    'doctor_id': [f'D{i:02d}' for i in range(1, 7)],
    'name':      [f'Dr_Smith_{i}' for i in range(1, 7)],
    'dept':      np.random.choice(['Cardiology', 'Neurology', 'Orthopedics'], size=6),
})

appointments = pd.DataFrame({
    'doctor_id': np.random.choice([f'D{i:02d}' for i in range(1, 7)], size=24),
    'wait_mins': np.random.randint(5, 60, size=24),
    'quarter':   np.random.choice(['Q1', 'Q2', 'Q3', 'Q4'], size=24),
})

feedback_h1 = pd.DataFrame({
    'doctor_id': np.random.choice([f'D{i:02d}' for i in range(1, 7)], size=8),
    'dept':      np.random.choice(['Cardiology', 'Neurology', 'Orthopedics'], size=8),
    'rating':    np.random.randint(1, 6, size=8),
    'half':      'H1',
})

feedback_h2 = pd.DataFrame({
    'doctor_id': np.random.choice([f'D{i:02d}' for i in range(1, 7)], size=8),
    'dept':      np.random.choice(['Cardiology', 'Neurology', 'Orthopedics'], size=8),
    'rating':    np.random.randint(1, 6, size=8),
    'half':      'H2',
})

# Your code here
ij = pd.merge(
    doctors, 
    appointments,
    on='doctor_id',
    how='inner'
)
ij['dept_avg_wait'] = ij.groupby('dept')['wait_mins'].transform('mean')
ij['above_avg_wait'] = ij['wait_mins']>ij['dept_avg_wait']
feedback = pd.concat(
    [feedback_h1, feedback_h2],
    ignore_index=True
)
feedback.pivot_table(
    index='dept',
    columns='half',
    values='rating'
)


half,H1,H2
dept,,
Cardiology,5.0,3.666667
Neurology,2.0,5.000000
Orthopedics,3.0,2.500000


---
## Level 4 — Real-world

### Task 7: Full Pipeline
Three DataFrames have been created for you: `items`, `stores`, and `transactions`.
Three helper functions have also been defined — use `.pipe()` to apply them as part of your pipeline.

1. Convert `txn_date` to datetime. Add `days_ago` = days from `txn_date` to `2026-03-19`. Find mean and median with NumPy.
2. Inner join `transactions` with `stores` on `store_id`. Inner join with `items` on `item_id`. Drop nulls.
3. Use `.pipe()` to apply all three functions in order: `add_revenue`, `flag_bulk`, `add_category_avg`.
4. Add `priority` (row-wise): `'High'` if `qty >= 5` **and** `is_bulk` is `True`, `'Medium'` if either is true, else `'Low'`
5. Pivot mean `revenue` by `city` (rows) and `category` (columns) using `pivot_table`. Stack the result.
6. Use `.xs()` to extract just `'Gadgets'` from the stacked result.
7. Find the correlation between `revenue` and `qty` using NumPy.

In [43]:
# Starter data — don't change this
np.random.seed(44)

items = pd.DataFrame({
    'item_id':   [f'IT{i:02d}' for i in range(1, 7)],
    'name':      ['Widget A', 'Widget B', 'Gadget X', 'Gadget Y', 'Tool 1', 'Tool 2'],
    'category':  ['Widgets', 'Widgets', 'Gadgets', 'Gadgets', 'Tools', 'Tools'],
    'unit_cost': [25, 40, 75, 120, 15, 30],
})

stores = pd.DataFrame({
    'store_id':  [f'ST{i:02d}' for i in range(1, 7)],
    'city':      np.random.choice(['Chicago', 'LA', 'NYC', 'Dallas'], size=6),
    'size_sqft': np.random.randint(1000, 5000, size=6),
})

transactions = pd.DataFrame({
    'txn_id':   [f'T{i:04d}' for i in range(1, 25)],
    'store_id': np.random.choice([f'ST{i:02d}' for i in range(1, 7)], size=24),
    'item_id':  np.random.choice([f'IT{i:02d}' for i in range(1, 7)], size=24),
    'qty':      np.random.randint(1, 10, size=24),
    'txn_date': pd.date_range('2024-01-01', periods=24, freq='15D').strftime('%Y-%m-%d'),
})

def add_revenue(df):
    df['revenue'] = df['unit_cost'] * df['qty']
    return df

def flag_bulk(df, threshold=200):
    df['is_bulk'] = df['revenue'] > threshold
    return df

def add_category_avg(df):
    df['category_avg'] = df.groupby('category')['revenue'].transform('mean')
    return df

# Your code here

transactions['txn_date'] = pd.to_datetime(transactions['txn_date'])
transactions['days_ago'] = (pd.to_datetime('2026-03-19') - transactions['txn_date']).dt.days
np.mean(transactions['days_ago'])
np.median(transactions['days_ago'])

ij = pd.merge(
    transactions,
    stores,
    on='store_id',
    how='inner'
)
ij2 = pd.merge(
    ij, 
    items,
    on='item_id',
    how='inner'
).dropna()

res = (
    ij2
    .pipe(add_revenue)
    .pipe(flag_bulk)
    .pipe(add_category_avg)
)
res['priority'] = res.apply(lambda row: 'High' if row['qty']>=5 and row['is_bulk']
                            else 'Medium' if row['qty']>=5 or row['is_bulk']
                            else 'Low', axis = 1)
ps = res.pivot_table(
    index = 'city',
    columns = 'category',
    values = 'revenue'
).stack().to_frame('mean_revenue')
ps
ps.xs('Gadgets',level='category')
np.corrcoef(res['revenue'], res['qty'])

array([[1.        , 0.41450938],
       [0.41450938, 1.        ]])